### ☀️ 참고 ☀️ 한글깨짐 방지

- 아래 코드셀 3개 실행 필수
- 윈도우, 맥 동일

❗️주의❗️
- Step1. 첫째 셀 실행
- Step2. '런타임 > 세션 다시시작'
- Step3. 이후 두번째 셀 실행

In [ ]:
# 나눔 폰트 설치
# 해당 셀 실행 후 '런타임 > 세션 다시 시작'

%%capture

!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl


# 나눔고딕 폰트 사용 설정 (폰트 설치 후 실행)
plt.rcParams["font.family"]="NanumGothic"

# 마이너스 깨짐 방지 (일반 하이픈(-)을 사용하게 설정)
mpl.rcParams['axes.unicode_minus'] = False

In [ ]:
# 한글 폰트 테스트 그래프

plt.plot([1, 2, 3], [10, -20, 30])
plt.title('한글 폰트 테스트')
plt.xlabel('x축')
plt.ylabel('y축')
plt.grid(True)
plt.show()

# 📘 CH09. 데이터 전처리 — 강의용 노트북

정규 4시간 수업 진행용입니다. Notion(md) 문서와 동일한 깊이로 구성했습니다.

> 📂 **실습 파일 안내**: 이번 챕터부터는 실제 분석 흐름 그대로 **CSV 파일을 직접 불러와서** 실습합니다.
> 함께 제공된 `orders.csv`(쇼핑몰 주문 데이터), `customers.csv`(고객 정보), `orders_extra.csv`(7월 추가 주문) 3개 파일을
> 노트북과 같은 위치에 업로드한 뒤 시작해주세요. (Colab: 왼쪽 📁 파일 탭 → 업로드)

---

# 📘 CH09-01. 결측치 처리


## 🤔 먼저 생각해보기

> **쇼핑몰 주문 데이터에서 몇몇 주문의 "배송평점"이 비어 있다면, 그 주문들을 통째로 버려야 할까요? 아니면 적절한 값으로 채워야 할까요?**

고객이 평점을 안 남겼거나 시스템 오류로 값이 빠지는 일은 실제 데이터에서 매우 흔합니다. 이런 **결측치(Missing Value)** 를 어떻게 다루는지가 분석 품질을 좌우합니다.

---

## 📖 핵심 개념

### 0) 실습 데이터 불러오기 — `pd.read_csv()`

한 온라인 쇼핑몰의 6월 주문 데이터입니다. 실제 데이터답게 **결측치, 이상치, 자료형 문제, 중복**이 곳곳에 숨어 있습니다. 이번 챕터 전체에서 이 파일을 사용합니다.

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")   # 노트북과 같은 위치에 orders.csv 업로드 필요
print(df.shape)                  # (305, 9) — 주문 305건, 컬럼 9개
df.head()
```

</details>

In [ ]:
# 코드 입력




### 1) 결측치란?

- 값이 존재하지 않는 상태를 의미하며, Pandas에서는 `NaN`(Not a Number)으로 표시됨
- 설문 미응답, 센서 오류, 데이터 수집 누락 등 다양한 이유로 발생

### 2) 결측치 확인 — `isnull()`/`isna()`, `notnull()`/`notna()`

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
print(df.isnull())          # 각 셀이 결측치인지 True/False
print(df.notnull())         # 반대: 값이 있으면 True

print(df.isnull().sum())    # 컬럼별 결측치 개수 — 카테고리 12, 수량 10, 배송평점 15
```

</details>

In [ ]:
# 코드 입력




### 3) 결측치 삭제 — `dropna()`

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
print(df.dropna())                     # 결측치가 하나라도 있는 "행"을 전부 삭제
print(df.dropna().shape)               # 305행 → 268행으로 줄어듦 (주문 37건이 통째로 사라짐!)
```

</details>

In [ ]:
# 결측치가 하나라도 있는 "행"을 전부 삭제



# 305행 → 268행으로 줄어듦 (주문 37건이 통째로 사라짐!)

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
print(df.dropna(axis=1))               # 결측치가 하나라도 있는 "컬럼"을 삭제
print(df.dropna(axis=1).columns)       # 카테고리, 수량, 배송평점 컬럼이 통째로 사라짐!
```

</details>

In [ ]:
# 결측치가 하나라도 있는 "컬럼"을 삭제




# 카테고리, 수량, 배송평점 컬럼이 통째로 사라짐!

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
print(df.dropna(how="all"))            # 모든 값이 결측치인 행만 삭제 (기본은 how="any")
print(df.dropna(how="all").shape)      # 전부 비어있는 행은 없으므로 305행 그대로
```

</details>

In [ ]:
# 모든 값이 결측치인 행만 삭제 (기본은 how="any")



# 전부 비어있는 행은 없으므로 305행 그대로

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
print(df.dropna(subset=["수량"]))       # 특정 컬럼(수량)에 결측치가 있는 행만 삭제
print(df.dropna(subset=["수량"]).shape) # 수량 결측 10건만 빠져 295행
```

</details>

In [ ]:
# 특정 컬럼(수량)에 결측치가 있는 행만 삭제



# 수량 결측 10건만 빠져 295행

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
print(df.dropna(thresh=8))             # 결측치가 아닌 값이 최소 8개 이상인 행만 유지
# 컬럼이 9개이므로 "결측치가 2개 이상인 행"은 탈락한다는 의미
```

</details>

In [ ]:
# 결측치가 아닌 값이 최소 8개 이상인 행만 유지
# 유효한 값이 7개 이하인 행 삭제
# 컬럼이 9개 (한 행에 들어갈 최대 값)이므로 "결측치가 2개 이상인 행"은 탈락한다는 의미

print(df.dropna(thresh=8))

### 4) 결측치 대체 — `fillna()`

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
print(df.fillna(0))                     # 모든 결측치를 0으로 채움 (단순하지만 위험할 수 있음)
# 카테고리 같은 문자열 컬럼에 0이 들어가면 이상해짐!
```

</details>

In [ ]:
# 모든 결측치를 0으로 채움 (단순하지만 위험할 수 있음)
# 원본 데이터에 들어가는 것은 아님 (연습을 위해 print로 확인)




# 카테고리 같은 문자열 컬럼에 0이 들어가면 이상해짐!

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
print(df["수량"].fillna(df["수량"].mean()))     # 평균값으로 채우기 (수치형에 자주 사용)
print(df["수량"].mean())                         # 평균 약 3.19 — 대부분이 1~5개인데 이상치(40~60)가 평균을 살짝 끌어올린 상태
```

</details>

In [ ]:
# 평균값으로 채우기 (수치형에 자주 사용)
# 평균 약 3.19 — 대부분이 1~5개인데 이상치(40~60)가 평균을 살짝 끌어올린 상태




<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
print(df["카테고리"].fillna(df["카테고리"].mode()[0]))   # 최빈값으로 채우기 (범주형에 자주 사용)
print(df["카테고리"].mode()[0])                            # 가장 자주 등장한 카테고리
```

</details>

In [ ]:
# 최빈값으로 채우기 (범주형에 자주 사용)
# mode()는 최빈값을 Series로 반환
# fillna()에 들어갈 것은 실제 값이므로 [0]으로 꺼내서 사용





<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 컬럼마다 다른 방식으로 한 번에 채우기
df_filled = df.fillna({
    "수량": df["수량"].mean(),
    "카테고리": df["카테고리"].mode()[0],
    "배송평점": df["배송평점"].median()
})
print(df_filled.isnull().sum())   # 모든 결측치가 사라짐
```

</details>

In [ ]:
# 컬럼마다 다른 방식으로 한 번에 채우기





<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 방향에 따라 채우기 — 시계열(주문일 순) 데이터 등에서 자주 사용
print(df["수량"].ffill())    # 앞의 값으로 채우기 (forward fill)
print(df["수량"].bfill())    # 뒤의 값으로 채우기 (backward fill)
```

</details>

In [ ]:
# 방향에 따라 채우기 — 시계열(주문일 순) 데이터 등에서 자주 사용
# 앞의 값으로 채우기 (forward fill)



# 뒤의 값으로 채우기 (backward fill)



### 5) 보간법 — `interpolate()`

앞뒤 값을 이용해 중간값을 수학적으로 추정해서 채우는 방법입니다. 연속적인 수치 데이터(시간에 따른 온도, 주가 등)에 특히 유용합니다.

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
temps = pd.Series([20, np.nan, np.nan, 26, 28])
print(temps.interpolate())   # [20. 22. 24. 26. 28.]  ← 앞뒤 값을 선형으로 이어서 추정
```

</details>

In [ ]:
# 시계열에서 많이 사용
# 앞뒤 값을 선형으로 이어서 추정




### 6) 서로 다른 방식 비교 — `dropna()` vs `fillna()`

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 방법 A: dropna() — 결측치가 있는 행을 통째로 삭제
result_a = df.dropna()
print("삭제 방식:", result_a.shape)          # 주문 37건이 사라짐 (데이터 손실)
```

</details>

In [ ]:
# 방법 A: dropna() — 결측치가 있는 행을 통째로 삭제




# 주문 37건이 사라짐 (데이터 손실)

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 방법 B: fillna() — 결측치를 적절한 값으로 채워 행을 보존
result_b = df.fillna({
    "수량": df["수량"].mean(),
    "카테고리": df["카테고리"].mode()[0],
    "배송평점": df["배송평점"].median()
})
print("채움 방식:", result_b.shape)          # 305건 모두 보존
```

</details>

In [ ]:
# 방법 B: fillna() — 결측치를 적절한 값으로 채워 행을 보존




# 305건 모두 보존

| 구분 | `dropna()` | `fillna()` |
|---|---|---|
| 데이터 손실 | 행/열이 통째로 사라짐 | 값만 채워지고 행은 보존됨 |
| 정보량 | 결측치가 많으면 데이터가 크게 줄어듦 | 원래 데이터 양을 유지 |
| 정확성 | 실제 존재했던 값은 아니므로 왜곡 없음 | 채워 넣은 값이 실제와 다를 수 있어 편향 위험 |
| 언제 사용? | 결측치 비율이 낮고, 데이터 양이 충분할 때 | 결측치 비율이 높거나, 행을 최대한 보존해야 할 때 |

> 💡 **핵심 요약**: 결측치가 적고 전체 데이터가 충분하면 `dropna()`로 간단히 제거해도 괜찮지만, 데이터가 귀하거나 결측치 비율이 높다면 `fillna()`로 합리적인 값을 채워 넣는 것이 정보 손실을 줄이는 방법입니다. 항상 "왜 이 값이 없는지"를 먼저 고민한 뒤 방법을 선택하세요.

---

## ⚠️ 자주 하는 실수

> ⚠️ `dropna()`를 기본 옵션 그대로 사용해서, 결측치가 조금이라도 있으면 그 행 전체가 삭제된다는 것을 놓치는 경우가 흔합니다.

> ⚠️ 수치형 컬럼의 결측치를 무조건 0으로 채워서, 평균 등의 통계치가 왜곡되는 경우가 있습니다 (0이 "없음"이 아니라 실제 값처럼 계산에 반영됨).

> ⚠️ `fillna()`나 `dropna()`를 실행한 결과를 변수에 다시 담지 않아서, 원본이 그대로인데 처리가 됐다고 착각하는 경우가 있습니다 (`inplace=True` 옵션을 쓰지 않는 한 원본은 안 바뀜).

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
df.fillna(0)          # ⚠️ 이 결과는 출력만 되고 df 자체는 안 바뀜!
df = df.fillna(0)       # ✅ 다시 대입해야 df가 실제로 바뀜
# df.fillna(0, inplace=True)   # ✅ 또는 inplace=True로 원본을 직접 수정
```

</details>

In [ ]:
# 코드 입력

df.fillna(0)          # ⚠️ 이 결과는 출력만 되고 df 자체는 안 바뀜!
df = df.fillna(0)       # ✅ 다시 대입해야 df가 실제로 바뀜
# df.fillna(0, inplace=True)   # ✅ 또는 inplace=True로 원본을 직접 수정


---

## 🚀 실무에서는?

> 🚀 실무에서는 결측치를 무작정 지우거나 채우기 전에, "왜 이 값이 없는지"(수집 오류인지, 애초에 응답을 안 한 건지 등)를 먼저 파악합니다. 수치형은 평균/중앙값, 범주형은 최빈값 또는 "미상"이라는 별도 카테고리로 채우는 것이 일반적인 실무 관행입니다.

---

## 🧩 Check Point

**Q1.** 결측치가 있는 행을 통째로 삭제하는 함수는? ① fillna() ② dropna() ③ isnull() ④ interpolate()
<details><summary>정답</summary>② dropna()</details>

**Q2.** 이상치의 영향을 적게 받는 대체값으로 적합한 것은? ① 평균 ② 중앙값 ③ 최댓값 ④ 0
<details><summary>정답</summary>② 중앙값(median)</details>

---

## 💻 실습

> 🔧 아래 코드 블록은 ipynb 셀로 그대로 변환됩니다. 난이도가 단계적으로 올라갑니다.

In [ ]:
# TODO 1 (확인): df의 컬럼별 결측치 개수를 확인하세요

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
print(df.isnull().sum())
```

</details>

In [ ]:
# 코드 입력




In [ ]:
# TODO 2 (삭제): 수량이 결측치인 행만 삭제하세요 (subset 활용)

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
print(df.dropna(subset=["수량"]))
```

</details>

In [ ]:
# 코드 입력




In [ ]:
# TODO 3 (평균 대체): 수량의 결측치를 수량 평균으로 채우세요

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
df["수량"] = df["수량"].fillna(df["수량"].mean())
print(df["수량"])
```

</details>

In [ ]:
# 코드 입력




In [ ]:
# TODO 4 (최빈값 대체): 카테고리의 결측치를 최빈값으로 채우세요

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
df["카테고리"] = df["카테고리"].fillna(df["카테고리"].mode()[0])
print(df["카테고리"])
```

</details>

In [ ]:
# 코드 입력




In [ ]:
# TODO 5 (보간법): [10, NaN, NaN, 40, 50] Series를 interpolate()로 채워보세요


<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
s = pd.Series([10, np.nan, np.nan, 40, 50])
print(s.interpolate())
```

</details>

In [ ]:
# 코드 입력




---

## 📝 실습 과제

In [ ]:
# 과제 1: 배송평점 컬럼의 결측치를 중앙값으로 채우고, 채우기 전후의 평균값 변화를 비교하세요

In [ ]:
# 과제 2: 수량, 카테고리, 배송평점 세 컬럼의 결측치를 각각 다른 방식(평균/최빈값/중앙값)으로
#         한 번에 채우는 딕셔너리를 fillna()에 전달하세요 (df를 새로 불러와서 시작: pd.read_csv)

In [ ]:
# 과제 3: dropna()로 결측치가 있는 행을 모두 제거했을 때와, fillna()로 채웠을 때
#         "수량" 평균이 어떻게 달라지는지 비교하세요

**예시 정답**

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 과제 1 예시 정답
before_mean = df["배송평점"].mean()
df["배송평점"] = df["배송평점"].fillna(df["배송평점"].median())
after_mean = df["배송평점"].mean()
print(f"채우기 전 평균: {before_mean:.2f}, 채운 후 평균: {after_mean:.2f}")
```

</details>

In [ ]:
# 과제 1 예시 정답





<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 과제 2 예시 정답
df = pd.read_csv("orders.csv")     # 원본을 새로 불러와서 시작
df_filled = df.fillna({
    "수량": df["수량"].mean(),
    "카테고리": df["카테고리"].mode()[0],
    "배송평점": df["배송평점"].median()
})
print(df_filled.isnull().sum())
```

</details>

In [ ]:
# 과제 2 예시 정답

df = pd.read_csv("orders.csv") # 원본을 새로 불러와서 시작





<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 과제 3 예시 정답
dropped_mean = df.dropna()["수량"].mean()
filled_mean = df["수량"].fillna(df["수량"].mean()).mean()
print(f"삭제 방식 평균: {dropped_mean:.2f}, 채움 방식 평균: {filled_mean:.2f}")
```

</details>

In [ ]:
# 과제 3 예시 정답






---

## 📌 핵심 정리

- 결측치 확인: `isnull()`/`isna()`, `notnull()`/`notna()` + `.sum()`으로 개수 집계
- 결측치 삭제: `dropna()` — `axis`, `how`, `subset`, `thresh` 옵션으로 세밀하게 제어
- 결측치 대체: `fillna()` — 평균/중앙값/최빈값/`ffill`/`bfill`, 컬럼별 딕셔너리로 한 번에 처리 가능
- `interpolate()`: 앞뒤 값을 이용한 수학적 추정으로 채우기 (연속 수치 데이터에 적합)
- 처리 결과는 재할당(`df = ...`) 또는 `inplace=True`가 있어야 실제로 반영됨

---

---

# 📘 CH09-02. 이상치 처리

## 🤔 먼저 생각해보기

> **300여 건의 주문 수량이 대부분 1~5개인데, 몇 건만 갑자기 40~60개라면 이 값들을 그대로 두고 "평균 주문 수량"을 계산해도 될까요?**

단체주문일 수도 있고 입력 실수일 수도 있습니다. 이렇게 다른 값들과 동떨어진 **이상치(Outlier)** 는 평균 같은 통계량을 크게 왜곡시키므로, 감지하고 판단하는 방법을 알아야 합니다.

---

## 📖 핵심 개념

### 0) 실습 데이터 준비

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")

# 지난 소단원(결측치)에서 배운 dropna()를 먼저 적용해 수량 데이터를 준비
qty = df["수량"].dropna()
print(qty.sort_values(ascending=False).head())   # 60, 50, 40... 수상하게 큰 값들이 보임
```

</details>

In [ ]:
# 코드 입력

import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")

# 지난 소단원(결측치)에서 배운 dropna()를 먼저 적용해 수량 데이터를 준비




### 1) 이상치란?

- 다른 값들과 비교했을 때 지나치게 크거나 작은 값
- 입력 실수(단위 오류 등), 특수한 상황(단체주문·VIP 대량구매 등), 데이터 수집 오류 등 다양한 원인으로 발생

### 2) `describe()`와 시각화로 이상치 감지하기

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
print(qty.describe())
# 50%(중앙값)는 3인데 max가 60?! — 대부분의 값과 동떨어진 최댓값은 이상치의 강력한 신호
# 300건쯤 되면 이상치 몇 개가 평균(3.19)을 눈에 안 띄게 왜곡하므로, describe로 꼭 점검해야 함
```

</details>

In [ ]:
# 코드 입력



# 50%(중앙값)는 3인데 max가 60?! — 대부분의 값과 동떨어진 최댓값은 이상치의 강력한 신호
# 300건쯤 되면 이상치 몇 개가 평균(3.19)을 눈에 안 띄게 왜곡하므로, describe로 꼭 점검해야 함

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
import matplotlib.pyplot as plt
import seaborn as sns

sns.boxplot(x=qty)
plt.title("주문 수량 분포 — 상자 밖의 점이 이상치 후보")
plt.show()
```

</details>

In [ ]:
# 코드 입력

import matplotlib.pyplot as plt
import seaborn as sns

# boxplot으로 확인
# x만 지정하면 가로로 출력됨 (y에 지정하면 세로 출력)





### 3) IQR(사분위수 범위) 방법

IQR은 데이터의 중간 50%가 퍼져있는 범위를 의미하며, 이를 기준으로 이상치를 판단하는 가장 널리 쓰이는 통계적 방법입니다.

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
Q1 = qty.quantile(0.25)    # 1사분위수 (하위 25% 지점) = 2.0
Q3 = qty.quantile(0.75)    # 3사분위수 (상위 25% 지점) = 4.0
IQR = Q3 - Q1              # 사분위수 범위 = 2.0

lower_bound = Q1 - 1.5 * IQR    # 하한선 = -1.0
upper_bound = Q3 + 1.5 * IQR    # 상한선 = 7.0
print(f"정상 범위: {lower_bound} ~ {upper_bound}")

outliers = qty[(qty < lower_bound) | (qty > upper_bound)]
print("이상치:", sorted(outliers.tolist()))    # [40, 50, 60] — 단체주문(?) 3건만 걸러짐
```

</details>

In [ ]:
# 코드 입력





### 4) Z-score(표준점수) 방법

Z-score는 "이 값이 평균에서 표준편차 몇 배만큼 떨어져 있는가"를 나타내는 지표입니다. 일반적으로 절댓값이 3을 넘으면 이상치로 간주합니다.

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
mean = qty.mean()
std = qty.std()

z_scores = (qty - mean) / std      # 각 값이 평균에서 표준편차 몇 배만큼 떨어졌는지
print(z_scores.tail())

print("Z-score 절댓값이 2를 넘는 값:", sorted(qty[z_scores.abs() > 2].tolist()))   # [40, 50, 60]
```

</details>

In [ ]:
# 코드 입력






print("Z-score 절댓값이 2를 넘는 값:", sorted(qty[z_scores.abs() > 2].tolist()))

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# scipy를 사용하면 더 간단하게 계산 가능
from scipy import stats

z = stats.zscore(qty)
print(sorted(qty[abs(z) > 2].tolist()))    # 같은 결과: [40, 50, 60]
```

</details>

In [ ]:
# scipy를 사용하면 더 간단하게 계산 가능





print(sorted(qty[abs(z) > 2].tolist()))

### 5) 이상치 제거 vs 이상치 대체(clip)

**방법 A: 이상치 제거**

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
clean_qty = qty[(qty >= lower_bound) & (qty <= upper_bound)]
print("제거 전:", len(qty), "건 / 제거 후:", len(clean_qty), "건")
print("제거 전 평균:", round(qty.mean(), 2), "→ 제거 후 평균:", round(clean_qty.mean(), 2))
# 이상치 3건 때문에 평균이 3.19 → 2.71로 달라짐 (300건 중 단 3건인데도!)
```

</details>

In [ ]:
# 이상치 제거 IQR 방식





**방법 B: 이상치 대체(clip) — 범위를 벗어난 값을 경계값으로 조정**

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
qty_clip = qty.clip(lower=lower_bound, upper=upper_bound)
print(qty_clip.sort_values(ascending=False).head())      # 40~60이 모두 상한선인 7.0으로 조정됨 (행은 그대로 보존)
print("clip 후 평균:", round(qty_clip.mean(), 2))
```

</details>

In [ ]:
# clip은 경계값 설정
# clip(lower , upper)





print(qty_clip.sort_values(ascending=False).head()) # 40~60이 모두 상한선인 7.0으로 조정됨 (행은 그대로 보존)
print("clip 후 평균:", round(qty_clip.mean(), 2))

### 6) 서로 다른 방식 비교 — IQR 방법 vs Z-score 방법

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 방법 A: IQR — 사분위수 기반, 극단적으로 치우친 분포에도 비교적 안정적
Q1, Q3 = qty.quantile(0.25), qty.quantile(0.75)
IQR = Q3 - Q1
iqr_outliers = qty[(qty < Q1 - 1.5*IQR) | (qty > Q3 + 1.5*IQR)]
print("IQR 방식 이상치:", iqr_outliers.tolist())
```

</details>

In [ ]:
# 방법 A: IQR — 사분위수 기반, 극단적으로 치우친 분포에도 비교적 안정적

Q1, Q3 = qty.quantile(0.25), qty.quantile(0.75)
IQR = Q3 - Q1

iqr_outliers = qty[(qty < Q1 - 1.5*IQR) | (qty > Q3 + 1.5*IQR)]
print("IQR 방식 이상치:", iqr_outliers.tolist())

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 방법 B: Z-score — 평균/표준편차 기반, 정규분포에 가까울 때 더 신뢰도 높음
z_scores = (qty - qty.mean()) / qty.std()
z_outliers = qty[z_scores.abs() > 2]
print("Z-score 방식 이상치:", z_outliers.tolist())
```

</details>

In [ ]:
# 방법 B: Z-score — 평균/표준편차 기반, 정규분포에 가까울 때 더 신뢰도 높음


z_scores = (qty - qty.mean()) / qty.std()
z_outliers = qty[z_scores.abs() > 2]
print("Z-score 방식 이상치:", z_outliers.tolist())

| 구분 | IQR 방법 | Z-score 방법 |
|---|---|---|
| 기준 | 사분위수(중간 50% 범위) | 평균과 표준편차 |
| 이상치 자체의 영향 | 상대적으로 덜 민감 (사분위수는 극단값에 강함) | 이상치가 평균/표준편차 자체를 왜곡시켜 오히려 판단이 부정확해질 수 있음 |
| 분포 가정 | 데이터 분포와 무관하게 적용 가능 | 정규분포에 가까울수록 신뢰도가 높음 |
| 언제 사용? | 분포를 모르거나 치우친 데이터일 때 (일반적으로 더 안전한 선택) | 데이터가 정규분포에 가깝다고 판단될 때 |

> 💡 **핵심 요약**: 어떤 방법이 "항상 옳다"기보다, 데이터의 분포 형태에 따라 선택합니다. 확신이 없다면 극단값 자체의 영향을 덜 받는 IQR 방법이 더 안전한 기본 선택지입니다.

---

## ⚠️ 자주 하는 실수

> ⚠️ 이상치를 무조건 "잘못된 데이터"로 여겨 무작정 삭제하는 경우가 있습니다 — 실제로 의미 있는 값(예: 진짜 단체주문)일 수도 있으므로, 삭제 전에 맥락을 확인해야 합니다.

> ⚠️ Z-score 계산 시, 이상치 자체가 평균과 표준편차를 왜곡시켜서 오히려 진짜 이상치를 놓치는 경우가 있습니다.

> ⚠️ IQR의 `1.5`라는 배수를 무조건 정답처럼 여기는 경우가 있는데, 이는 관례적인 기준일 뿐 상황에 따라 조정할 수 있습니다 (더 엄격하게 하려면 3을 사용하기도 함).

---

## 🚀 실무에서는?

> 🚀 실무에서는 이상치를 "무조건 제거"하기보다, 그 이상치가 실제 오류인지 아니면 의미 있는 특수 사례인지(VIP 고객, 특수 거래 등)를 먼저 확인합니다. 머신러닝 모델링(CH03) 전에는 이상치가 모델 성능에 미치는 영향을 고려해 제거·대체 여부를 결정합니다.

---

## 🧩 Check Point

**Q1.** IQR 방법에서 이상치의 상한선을 구하는 공식은? ① Q3 + IQR ② Q3 + 1.5×IQR ③ Q1 - 1.5×IQR ④ 평균 + 표준편차
<details><summary>정답</summary>② Q3 + 1.5×IQR</details>

**Q2.** 값을 범위 밖으로 나가지 않도록 경계값으로 조정(대체)하는 함수는? ① dropna() ② clip() ③ fillna() ④ astype()
<details><summary>정답</summary>② clip()</details>

---

## 💻 실습

> 🔧 아래 코드 블록은 ipynb 셀로 그대로 변환됩니다. 난이도가 단계적으로 올라갑니다.

In [ ]:
# TODO 1 (감지): qty(수량)의 describe()를 확인해 이상치가 의심되는지 살펴보세요

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
print(qty.describe())
```

</details>

In [ ]:
# 코드 입력




In [ ]:
# TODO 2 (IQR 계산): 수량의 Q1, Q3, IQR, 상한/하한선을 각각 계산하세요

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
Q1 = qty.quantile(0.25)
Q3 = qty.quantile(0.75)
IQR = Q3 - Q1
print("Q1:", Q1, "Q3:", Q3, "IQR:", IQR)
print("하한:", Q1 - 1.5*IQR, "상한:", Q3 + 1.5*IQR)
```

</details>

In [ ]:
# 코드 입력




In [ ]:
# TODO 3 (이상치 찾기): IQR 기준으로 이상치에 해당하는 "주문 행 전체"를 df에서 출력하세요

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
lower = Q1 - 1.5*IQR
upper = Q3 + 1.5*IQR
print(df[(df["수량"] < lower) | (df["수량"] > upper)])   # 어떤 주문이었는지 확인
```

</details>

In [ ]:
# 코드 입력




In [ ]:
# TODO 4 (제거 vs clip): 이상치를 제거한 결과와 clip()으로 대체한 결과의 평균을 비교하세요

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
removed = qty[(qty >= lower) & (qty <= upper)]
clipped = qty.clip(lower=lower, upper=upper)
print("제거 방식 평균:", round(removed.mean(), 2))
print("clip 방식 평균:", round(clipped.mean(), 2))
```

</details>

In [ ]:
# 코드 입력




---

## 📝 실습 과제

In [ ]:
# 과제 1: 배송평점 컬럼에서 (50, 55가 이상치로 의심됨)
#         IQR 방법으로 이상치를 찾아 해당 주문 행을 출력하세요 (결측치는 dropna로 먼저 제거)

In [ ]:
# 과제 2: 배송평점의 Z-score를 계산해서 절댓값 2를 넘는 값을 이상치로 판별하고,
#         IQR 방법의 결과와 같은지 비교하세요

**예시 정답**

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 과제 1 예시 정답
rating = df["배송평점"].dropna()
Q1, Q3 = rating.quantile(0.25), rating.quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
print(f"정상 범위: {lower} ~ {upper}")
print(df[(df["배송평점"] < lower) | (df["배송평점"] > upper)])   # 평점 50·55짜리 주문 — 오타로 추정
```

</details>

In [ ]:
# 과제 1 예시 정답





<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 과제 2 예시 정답
z = (rating - rating.mean()) / rating.std()
print("Z-score 방식:", sorted(rating[z.abs() > 2].tolist()))    # [50, 55] — IQR 방법과 동일한 결과
```

</details>

In [ ]:
# 과제 2 예시 정답





---

## 📌 핵심 정리

- 이상치는 `describe()`와 박스플롯으로 1차 감지 가능
- IQR 방법: `Q1 - 1.5×IQR` ~ `Q3 + 1.5×IQR` 범위를 벗어나면 이상치
- Z-score 방법: 평균에서 표준편차 대비 몇 배 떨어져 있는지 계산, 보통 절댓값 3 기준
- 이상치는 제거(행 삭제)하거나 `clip()`으로 경계값으로 대체 가능
- IQR은 분포에 덜 민감, Z-score는 정규분포에 가까울 때 더 신뢰도 높음

---

---

# 📘 CH09-03. 자료형 변환 (중복 처리 포함)

## 🤔 먼저 생각해보기

> **CSV에서 불러온 "단가" 컬럼이 "89,000"처럼 콤마가 들어간 문자열이라면, 합계나 평균 계산이 될까요?**

콤마 때문에 숫자가 아닌 **문자열(object)** 로 읽혀서 계산이 불가능합니다. 실무 데이터의 상당수가 이런 상태로 들어오기 때문에, **자료형 변환**은 전처리의 필수 단계입니다.

---

## 📖 핵심 개념

### 0) 실습 데이터 준비 — 자료형부터 확인

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
print(df.dtypes)
# 단가가 int가 아니라 object(문자열)! — "89,000"의 콤마 때문
# 주문일도 진짜 날짜가 아니라 object(문자열)
```

</details>

In [ ]:
# 코드 입력

import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
print(df.dtypes)

# 단가가 int가 아니라 object(문자열)! — "89,000"의 콤마 때문
# 주문일도 진짜 날짜가 아니라 object(문자열)

### 1) 자료형 변환이 필요한 이유

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
print(df["단가"].sum())   # 문자열끼리 이어붙여짐! 숫자 합계가 아님
# "89,00018,00039,000..." 같은 긴 문자열이 출력됨
```

</details>

In [ ]:
# 코드 입력





### 2) `astype()` — 자료형을 명시적으로 지정해서 변환

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 순수한 숫자 문자열이라면 astype()으로 바로 변환 가능
s = pd.Series(["1000", "2000", "3000"])
print(s.astype(int))
print(s.astype(int).sum())    # 6000 — 이제 진짜 숫자 합계

# 하지만 우리 데이터는...
# df["단가"].astype(int)      # ❌ "89,000"의 콤마 때문에 에러 발생!
```

</details>

In [ ]:
# 순수한 숫자 문자열이라면 astype()으로 바로 변환 가능





# 하지만 우리 데이터는...
# df["단가"].astype(int) # ❌ "89,000"의 콤마 때문에 에러 발생!

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 콤마를 먼저 제거한 뒤 변환 — str.replace()와의 조합 (실무 표준 패턴)
df["단가"] = df["단가"].str.replace(",", "").astype(int)
print(df["단가"].dtype)     # int64
print(df["단가"].sum())     # 이제 제대로 된 합계가 계산됨
```

</details>

In [ ]:
# 콤마를 먼저 제거한 뒤 변환 — str.replace()와의 조합 (실무 표준 패턴)





> ⚠️ `astype(int)`는 변환하려는 값이 전부 숫자로 해석 가능해야 합니다. "4200원"처럼 숫자가 아닌 문자가 섞여 있으면 에러가 발생합니다.

### 3) `pd.to_numeric()` — 숫자 변환에 특화, 에러 처리 옵션 제공

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
messy = pd.Series(["100", "200", "오류", "400"])
# print(messy.astype(int))                     # ❌ "오류" 때문에 에러 발생
# errors 옵션은 raise, coerce, ignore

print(pd.to_numeric(messy, errors="coerce"))    # 변환 안 되는 값은 NaN으로 처리
# 0    100.0
# 1    200.0
# 2      NaN
# 3    400.0
```

</details>

In [ ]:
# 코드 입력





**문자열 정리 — `str.strip()` (변환 전 필수 청소)**

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
print(df["카테고리"].unique())
# ' 전자기기', '생활용품 ' 처럼 앞뒤 공백이 섞인 값이 별도의 카테고리로 취급되고 있음!

df["카테고리"] = df["카테고리"].str.strip()      # 양쪽 공백 제거
print(df["카테고리"].unique())                    # 4개 카테고리로 정상적으로 합쳐짐
print(df["카테고리"].value_counts())
```

</details>

In [ ]:
# 코드 입력





> 💡 엑셀·CSV 데이터에는 눈에 보이지 않는 앞뒤 공백이 섞여 있는 경우가 매우 흔합니다. `" 전자기기"`와 `"전자기기"`는 다른 값으로 취급되어 `value_counts()`나 `groupby()` 결과가 이상하게 쪼개지므로, 문자열 컬럼은 **`str.strip()`으로 먼저 청소한 뒤** `str.replace()` → `astype()`/`to_numeric()` 순서로 변환하는 것이 표준 패턴입니다.

### 4) `pd.to_datetime()` — 날짜 문자열을 진짜 날짜 자료형으로

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
df["주문일"] = pd.to_datetime(df["주문일"])
print(df["주문일"].dtype)          # datetime64 — 진짜 날짜 자료형

# 날짜가 되면 년/월/일/요일을 자유롭게 추출 가능
print(df["주문일"].dt.month.head())      # 월
print(df["주문일"].dt.day_name().head()) # 요일 이름
```

</details>

In [ ]:
# 코드 입력





### 5) `replace()` — 특정 값을 다른 값으로 치환

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 결제수단코드 C/T/P를 알아보기 쉬운 이름으로
df["결제수단"] = df["결제수단코드"].replace({"C": "카드", "T": "계좌이체", "P": "간편결제"})
print(df[["결제수단코드", "결제수단"]].head())
```

</details>

In [ ]:
# 결제수단코드 C/T/P를 알아보기 쉬운 이름으로





<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 값 하나만 바꿀 때
s = pd.Series([1, 2, 3, 2, 1])
print(s.replace(2, 200))   # 값 2를 200으로 치환
```

</details>

In [ ]:
# 값 하나만 바꿀 때





### 6) `map()` — Series 전체에 매핑 규칙 적용

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
pay_map = {"C": "카드", "T": "계좌이체", "P": "간편결제"}
df["결제수단_map"] = df["결제수단코드"].map(pay_map)
print(df[["결제수단코드", "결제수단_map"]].head())
```

</details>

In [ ]:
# 방법1 (변수에 먼저 저장)
pay_map = {"C": "카드", "T": "계좌이체", "P": "간편결제"}




# 방법2 (바로 작성)





<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# map()에 함수를 전달할 수도 있음
df["결제수단코드_소문자"] = df["결제수단코드"].map(lambda x: x.lower())
print(df["결제수단코드_소문자"].head())
```

</details>

In [ ]:
# map()에 함수를 전달할 수도 있음
# 단, lambda는 조건 하나인 경우에만 사용





### 7) `rename()` — 컬럼명·인덱스명 변경

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
df = df.rename(columns={"단가": "상품단가", "배송평점": "평점"})
print(df.columns)
```

</details>

In [ ]:
# 코드 입력





<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
df_renamed_index = df.rename(index={0: "첫주문"})
print(df_renamed_index.head(2))
```

</details>

In [ ]:
# 코드 입력





### 8) 중복 확인 및 제거 — `duplicated()`, `drop_duplicates()`

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
print(df.duplicated())          # 각 행이 이전에 나온 행과 완전히 같은지(중복인지)
print(df.duplicated().sum())    # 중복 주문이 5건! (시스템 오류로 두 번 기록된 주문)
```

</details>

In [ ]:
# 각 행이 이전에 나온 행과 완전히 같은지(중복인지)



# 중복 주문이 5건! (시스템 오류로 두 번 기록된 주문)




<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
df_no_dup = df.drop_duplicates()    # 중복 행 제거 (기본은 첫 번째 값만 남김)
print("제거 전:", df.shape, "→ 제거 후:", df_no_dup.shape)   # 305건 → 300건
```

</details>

In [ ]:
# 중복 행 제거 (기본은 첫 번째 값만 남김)
# keep = 'last' : 마지막행만 남김





<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
print(df.duplicated(subset=["주문번호"]))          # "주문번호"만 기준으로 중복 확인
print(df[df.duplicated(subset=["주문번호"], keep=False)])   # 중복된 주문 전체 확인 (keep=False: 원본까지 표시)
```

</details>

In [ ]:
# "주문번호"만 기준으로 중복 확인



# 중복된 주문 전체 확인 (keep=False: 첫번째 중복까지 True로)





### 9) 서로 다른 방식 비교 — `map()` vs `replace()`

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
codes = df["결제수단코드"]

# 방법 A: map() — 매핑에 없는 값은 NaN이 됨
print(codes.map({"C": "카드", "T": "계좌이체"}).head())    # P는 매핑에 없어서 NaN!

# 방법 B: replace() — 매핑에 없는 값은 원래 값 그대로 유지
print(codes.replace({"C": "카드", "T": "계좌이체"}).head())  # P는 "P" 그대로
```

</details>

In [ ]:
# 방법 A: map() — 매핑에 없는 값은 NaN이 됨



# 방법 B: replace() — 매핑에 없는 값은 원래 값 그대로 유지





| 구분 | `map()` | `replace()` |
|---|---|---|
| 매핑에 없는 값 | `NaN`으로 바뀜 | 원래 값 그대로 유지 |
| 사용 대상 | Series 전용 | Series와 DataFrame 모두 사용 가능 |
| 용도 | 전체 값을 새로운 값 체계로 완전히 바꿀 때 | 일부 특정 값만 콕 집어 바꾸고 나머지는 그대로 둘 때 |

> 💡 **핵심 요약**: 모든 값을 빠짐없이 다른 값으로 매핑해야 한다면(그리고 누락을 발견하고 싶다면) `map()`을, 일부 값만 살짝 수정하고 나머지는 그대로 두고 싶다면 `replace()`를 사용하세요.

---

## ⚠️ 자주 하는 실수

> ⚠️ `astype(int)`를 콤마(`"4,200"`)나 단위(`"4200원"`)가 포함된 문자열에 그대로 사용해서 에러가 나는 경우가 흔합니다 — 먼저 문자열을 정리한 뒤 변환해야 합니다.

> ⚠️ `map()`을 사용할 때 매핑 딕셔너리에 없는 값이 조용히 `NaN`으로 바뀌는 것을 눈치채지 못하는 경우가 있습니다.

> ⚠️ `drop_duplicates()`의 기본 동작(`keep='first'`)을 모르고, 어떤 중복 행이 남는지 예상과 다르게 나오는 경우가 있습니다.

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 종합 연습: 원본을 다시 불러와 전처리 파이프라인을 한 번에
df = pd.read_csv("orders.csv")
df["단가"] = df["단가"].str.replace(",", "").astype(int)     # 콤마 제거 + 숫자 변환
df["카테고리"] = df["카테고리"].str.strip()                    # 공백 청소
df["주문일"] = pd.to_datetime(df["주문일"])                    # 날짜 변환
df = df.drop_duplicates()                                     # 중복 제거
print(df.dtypes)
print(df.shape)
```

</details>

In [ ]:
# 종합 연습: 원본을 다시 불러와 전처리 파이프라인을 한 번에
df = pd.read_csv("orders.csv")

df["단가"] = df["단가"].str.replace(",", "").astype(int)# 콤마 제거 + 숫자 변환
df["카테고리"] = df["카테고리"].str.strip()  # 공백 청소
df["주문일"] = pd.to_datetime(df["주문일"])   # 날짜 변환

df = df.drop_duplicates()  # 중복 제거
print(df.dtypes)
print(df.shape)

---

## 🚀 실무에서는?

> 🚀 실무 데이터의 상당수는 엑셀/CSV/API에서 숫자가 문자열로, 날짜가 텍스트로 들어옵니다. 분석을 시작하기 전 `astype()`, `to_numeric()`, `to_datetime()`으로 자료형을 정리하는 것이 거의 항상 첫 번째 전처리 단계이며, 중복 데이터 제거는 회원 데이터, 거래 내역 등에서 필수적입니다.

---

## 🧩 Check Point

**Q1.** 문자열을 숫자로 변환하되, 변환 불가능한 값은 에러 대신 NaN으로 처리하고 싶을 때 사용하는 함수는? ① astype() ② pd.to_numeric(errors='coerce') ③ replace() ④ map()
<details><summary>정답</summary>② pd.to_numeric(errors='coerce')</details>

**Q2.** 매핑 딕셔너리에 없는 값을 NaN으로 만드는 함수는? ① replace() ② map() ③ rename() ④ astype()
<details><summary>정답</summary>② map()</details>

---

## 💻 실습

> 🔧 아래 코드 블록은 ipynb 셀로 그대로 변환됩니다. 난이도가 단계적으로 올라갑니다.

In [ ]:
# TODO 1 (astype): orders.csv를 새로 불러와 "단가"를 콤마 제거 후 정수형으로 변환하세요

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
df = pd.read_csv("orders.csv")
df["단가"] = df["단가"].str.replace(",", "").astype(int)
print(df["단가"].dtype)
print(df["단가"].mean())
```

</details>

In [ ]:
# 코드 입력




In [ ]:
# TODO 2 (to_datetime): "주문일"을 datetime으로 변환하고 요일 이름을 추출하세요

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
df["주문일"] = pd.to_datetime(df["주문일"])
print(df["주문일"].dt.day_name())
```

</details>

In [ ]:
# 코드 입력




In [ ]:
# TODO 3 (replace): "결제수단코드"의 C/T/P를 각각 "카드"/"계좌이체"/"간편결제"로 replace하세요

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
print(df["결제수단코드"].replace({"C": "카드", "T": "계좌이체", "P": "간편결제"}))
```

</details>

In [ ]:
# 코드 입력




In [ ]:
# TODO 4 (중복 확인/제거): df에서 중복 주문 개수를 확인하고 제거하세요

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
print(df.duplicated().sum())
df = df.drop_duplicates()
print(df.shape)
```

</details>

In [ ]:
# 코드 입력




In [ ]:
# TODO 5 (rename): "배송평점" 컬럼명을 "평점"으로 변경하세요

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
df = df.rename(columns={"배송평점": "평점"})
print(df.columns)
```

</details>

In [ ]:
# 코드 입력




---

## 📝 실습 과제

In [ ]:
# 과제 1: "카테고리" 컬럼의 공백을 str.strip()으로 정리한 뒤,
#         정리 전후의 value_counts() 결과를 비교하세요 (df를 새로 불러와서 시작)

In [ ]:
# 과제 2: "주문번호" 기준 중복 주문을 찾아 제거하되, 가장 마지막 기록을 남기세요 (keep="last")

In [ ]:
# 과제 3: 고객 등급 코드 ["V","G","N"]을 map()으로 각각 "VIP","일반","신규"로 변환하고,
#         매핑에 없는 코드 "X"가 섞였을 때 map()과 replace()의 결과 차이를 확인하세요

**예시 정답**

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 과제 1 예시 정답
df = pd.read_csv("orders.csv")
print("정리 전:")
print(df["카테고리"].value_counts())     # ' 전자기기' 등이 따로 집계됨

df["카테고리"] = df["카테고리"].str.strip()
print("정리 후:")
print(df["카테고리"].value_counts())     # 4개 카테고리로 깔끔하게 합쳐짐
```

</details>

In [ ]:
# 과제 1 예시 정답




<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 과제 2 예시 정답
dedup = df.drop_duplicates(subset=["주문번호"], keep="last")
print("제거 전:", df.shape, "→ 제거 후:", dedup.shape)
```

</details>

In [ ]:
# 과제 2 예시 정답





<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 과제 3 예시 정답
grades = pd.Series(["V", "G", "N", "X"])
grade_map = {"V": "VIP", "G": "일반", "N": "신규"}
print(grades.map(grade_map))       # X는 매핑에 없어서 NaN
print(grades.replace(grade_map))   # X는 "X" 그대로 유지
```

</details>

In [ ]:
# 과제 3 예시 정답






---

## 📌 핵심 정리

- `astype()`: 자료형을 명시적으로 변환 / `pd.to_numeric(errors='coerce')`: 숫자 변환 실패 시 NaN 처리
- `pd.to_datetime()`: 날짜 문자열 변환, `.dt` 접근자로 연/월/일 등 추출
- `replace()`: 특정 값만 치환(나머지 유지) / `map()`: 전체를 매핑(없으면 NaN)
- `rename(columns={}, index={})`: 컬럼명/인덱스명 변경
- `duplicated()`: 중복 여부 확인 / `drop_duplicates(subset=, keep=)`: 중복 제거
- 문자열 변환 전 `str.strip()`으로 보이지 않는 공백 먼저 제거 (→ `str.replace()` → 형변환 순)

---

---

# 📘 CH09-04. 데이터 병합 (피벗테이블·JSON 포함)

## 🤔 먼저 생각해보기

> **"주문 데이터"에는 고객ID만 있고, "고객 명단"에는 고객의 이름·등급·도시가 있습니다. 각 주문이 어떤 등급의 고객에게서 왔는지 알려면 어떻게 해야 할까요?**

실무 데이터는 이렇게 여러 테이블로 나뉘어 저장됩니다. 두 표를 **기준 키(고객ID)** 로 연결하는 것이 데이터 병합입니다.

---

## 📖 핵심 개념

### 0) 실습 데이터 준비 — 주문 + 고객 두 개의 표

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
import pandas as pd

# 주문 데이터 (지난 소단원에서 배운 전처리를 먼저 적용)
orders = pd.read_csv("orders.csv")
orders["단가"] = orders["단가"].str.replace(",", "").astype(int)
orders["카테고리"] = orders["카테고리"].str.strip()
orders = orders.drop_duplicates()

# 고객 데이터 (새 파일!)
customers = pd.read_csv("customers.csv")
print(customers)
# 주의: 주문에는 고객 명단에 없는 "C999"가 있고, 명단에는 6월 주문이 없는 고객(C028~C030)도 있음
```

</details>

In [ ]:
# 코드 입력

import pandas as pd

# 주문 데이터 불러오기



# 고객 데이터 불러오기 (새 파일!)




# 지난 소단원에서 배운 전처리를 먼저 적용
orders["단가"] = orders["단가"].str.replace(",", "").astype(int)
orders["카테고리"] = orders["카테고리"].str.strip()
orders = orders.drop_duplicates()




# 주의: 주문에는 고객 명단에 없는 "C999"가 있고, 명단에는 6월 주문이 없는 고객(C028~C030)도 있음

### 1) 왜 데이터를 병합해야 할까?

- 실무 데이터는 회원 정보, 주문 정보, 상품 정보처럼 여러 테이블로 나뉘어 저장되는 경우가 대부분 (데이터베이스 설계 원칙)
- 분석하려면 필요한 정보를 하나의 표로 모아야 함

### 2) `pd.concat()` — 단순히 이어붙이기

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 세로로 이어붙이기 (행 방향, axis=0) — 같은 컬럼 구조의 표를 위아래로
extra = pd.read_csv("orders_extra.csv")     # 7월 초 추가 주문 (새 파일!)
extra["단가"] = extra["단가"].str.replace(",", "").astype(int)

all_orders = pd.concat([orders, extra], ignore_index=True)   # ignore_index: 인덱스를 0부터 다시 매김
print("6월 주문:", orders.shape, "+ 7월 주문:", extra.shape, "→ 합침:", all_orders.shape)
```

</details>

In [ ]:
# 세로로 이어붙이기 (행 방향, axis=0)
# 같은 컬럼 구조의 표를 위아래로

# 7월 초 추가 주문 (새 파일!)
extra = pd.read_csv("orders_extra.csv")





<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 가로로 이어붙이기 (열 방향, axis=1) — 같은 행 순서를 가진 표를 옆으로
info_a = pd.DataFrame({"상품명": ["무선이어폰", "텀블러"]})
info_b = pd.DataFrame({"재고": [12, 30]})

combined_h = pd.concat([info_a, info_b], axis=1)
print(combined_h)
```

</details>

In [ ]:
# 가로로 이어붙이기 (열 방향, axis=1) — 같은 행 순서를 가진 표를 옆으로





### 3) `pd.merge()` — 특정 컬럼(기준 키)으로 정확하게 결합

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
result = pd.merge(orders, customers, on="고객ID")   # "고객ID" 값이 일치하는 행끼리 결합
print(result[["주문번호", "고객ID", "고객명", "등급", "상품명"]].head())
# 이제 각 주문에 고객 이름과 등급이 붙었다!
```

</details>

In [ ]:
# "고객ID" 값이 일치하는 행끼리 결합





### 4) `merge()`의 `how` 옵션 — 4가지 결합 방식

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# inner (기본값): 양쪽 모두에 키가 있는 행만
inner = pd.merge(orders, customers, on="고객ID", how="inner")
print("inner:", inner.shape)     # C999 주문은 명단에 없어서 탈락

# left: 왼쪽(orders)은 전부 유지, 매칭 안 되면 NaN
left = pd.merge(orders, customers, on="고객ID", how="left")
print("left:", left.shape)       # C999 주문도 유지 (고객명은 NaN)
print(left[left["고객명"].isnull()][["주문번호", "고객ID"]])   # 명단에 없는 고객의 주문 확인

# right: 오른쪽(customers)은 전부 유지
right = pd.merge(orders, customers, on="고객ID", how="right")
print("right:", right.shape)     # 6월 주문이 없는 고객(C028~C030)도 등장 (주문 정보는 NaN)

# outer: 양쪽 모두 유지
outer = pd.merge(orders, customers, on="고객ID", how="outer")
print("outer:", outer.shape)
```

</details>

In [ ]:
# inner (기본값): 양쪽 모두에 키가 있는 행만




# C999 주문은 명단에 없어서 탈락

In [ ]:
# left: 왼쪽(orders)은 전부 유지, 매칭 안 되면 NaN



# C999 주문도 유지 (고객명은 NaN)

In [ ]:
# right: 오른쪽(customers)은 전부 유지



# 6월 주문이 없는 고객(C028~C030)도 등장 (주문 정보는 NaN)

In [ ]:
# outer: 양쪽 모두 유지





| `how` 옵션 | 결과 | 비유 |
|---|---|---|
| `inner` (기본) | 양쪽에 공통으로 있는 것만 | 교집합 |
| `left` | 왼쪽 표의 모든 행 + 매칭되는 오른쪽 정보 | 왼쪽 기준 |
| `right` | 오른쪽 표의 모든 행 + 매칭되는 왼쪽 정보 | 오른쪽 기준 |
| `outer` | 양쪽의 모든 행 (매칭 안 되면 NaN) | 합집합 |

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 컬럼명이 서로 다를 때 — left_on, right_on
customers2 = customers.rename(columns={"고객ID": "회원번호"})
result2 = pd.merge(orders, customers2, left_on="고객ID", right_on="회원번호")
print(result2[["주문번호", "고객ID", "회원번호", "고객명"]].head())
```

</details>

In [ ]:
# 컬럼명이 서로 다를 때 — left_on, right_on

#고객ID를 회원번호로 변경



# 왼쪽은 고객ID, 오른쪽은 회원번호를 기준으로 합침





### 5) `join()` — 인덱스를 기준으로 병합

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
orders_indexed = orders.set_index("고객ID")
customers_indexed = customers.set_index("고객ID")

joined = orders_indexed.join(customers_indexed)    # 인덱스(고객ID)를 기준으로 병합
print(joined[["주문번호", "상품명", "고객명", "등급"]].head())
```

</details>

In [ ]:
# orders,customers의 인덱스를 '고객ID'로 변경
orders_indexed = orders.set_index("고객ID")
customers_indexed = customers.set_index("고객ID")

# 인덱스(고객ID)를 기준으로 병합




### 6) 피벗테이블 복습 및 심화 — `pivot_table()`

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 카테고리별 × 결제수단별 주문 수량 합계
pivot1 = orders.pivot_table(index="카테고리", columns="결제수단코드", values="수량", aggfunc="sum")
print(pivot1)
```

</details>

In [ ]:
# 카테고리별 × 결제수단별 주문 수량 합계





<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 여러 집계함수를 동시에, 합계 행/열도 추가 (margins)
pivot2 = orders.pivot_table(index="카테고리", values=["수량", "단가"],
                            aggfunc={"수량": "sum", "단가": "mean"},
                            margins=True)      # margins: "All" 합계 행 추가
print(pivot2)
```

</details>

In [ ]:
# 여러 집계함수를 동시에, 합계 행/열도 추가 (margins)
# margins: 하단에 "All" 합계 행 추가





### 7) JSON 데이터 다루기

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
import json

# 파이썬 딕셔너리를 JSON 문자열로/에서 변환
order_json = {
    "주문번호": 3001,
    "상품명": "무선이어폰",
    "고객": {"고객ID": "C001", "등급": "VIP"}
}
json_str = json.dumps(order_json, ensure_ascii=False)   # 딕셔너리 → JSON 문자열
print(json_str)

parsed = json.loads(json_str)                            # JSON 문자열 → 딕셔너리
print(parsed["고객"]["등급"])
```

</details>

In [ ]:
# 코드 입력

import json

# 딕셔너리 생성
order_json = {
    "주문번호": 3001,
    "상품명": "무선이어폰",
    "고객": {"고객ID": "C001", "등급": "VIP"}
}

# 딕셔너리 → JSON 문자열(String)으로 변환
# 기본형: json.dumps()
# ensure_ascii=False (한글 깨지지 않게)
# ensure_ascii = True는 모든 문자열을 ASCII 문자로만 표현
# 이 때 한글은 \uXXXX 형태의 유니코드 이스케이프 문자로 변환됨






<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 여러 개의 중첩 JSON 레코드를 한 번에 정규화 (쇼핑몰 API 응답 형태)
records = [
    {"주문번호": 3001, "상품명": "무선이어폰", "고객": {"고객ID": "C001", "등급": "VIP"}},
    {"주문번호": 3002, "상품명": "텀블러",     "고객": {"고객ID": "C002", "등급": "일반"}},
    {"주문번호": 3003, "상품명": "후드티",     "고객": {"고객ID": "C004", "등급": "신규"}},
]
df_json = pd.json_normalize(records)    # 중첩 구조가 "고객.고객ID"처럼 평평한 컬럼으로
print(df_json)
```

</details>

In [ ]:
# 여러 개의 중첩 JSON 레코드를 한 번에 정규화 (쇼핑몰 API 응답 형태)
records = [
    {"주문번호": 3001, "상품명": "무선이어폰", "고객": {"고객ID": "C001", "등급": "VIP"}},
    {"주문번호": 3002, "상품명": "텀블러", "고객": {"고객ID": "C002", "등급": "일반"}},
    {"주문번호": 3003, "상품명": "후드티", "고객": {"고객ID": "C004", "등급": "신규"}},
]

# 중첩 구조가 "고객.고객ID"처럼 평평한 컬럼으로
# 즉, 딕셔너리 안에 딕셔너리가 밖으로 나와서 같은 레벨로





<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# DataFrame ↔ JSON 파일 저장/불러오기
df_json.to_json("orders_sample.json", orient="records", force_ascii=False)
loaded = pd.read_json("orders_sample.json")
print(loaded)
```

</details>

In [ ]:
# DataFrame ↔ JSON 파일 저장/불러오기
# to_json() : DataFrame을 json 형태로 저장
# orient = 'records' (한 행을 json 객체 하나로 저장
# force_ascii = False (한글을 그대로 저장)






### 8) 서로 다른 방식 비교 — `merge()` vs `concat()`

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# concat(): 단순히 이어붙임 (기준 키로 매칭하지 않음, 구조가 같다고 가정)
result_concat = pd.concat([orders, extra], ignore_index=True)
print("concat:", result_concat.shape)     # 행이 그냥 아래로 늘어남
```

</details>

In [ ]:
# concat(): 단순히 이어붙임 (기준 키로 매칭하지 않음, 구조가 같다고 가정)

print(orders.shape)
print(extra.shape)





<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# merge(): 특정 컬럼(키) 값이 일치하는 것끼리 정확하게 매칭
result_merge = pd.merge(orders, customers, on="고객ID")
print("merge:", result_merge.shape)       # 컬럼이 옆으로 늘어남 (고객 정보가 붙음)
```

</details>

In [ ]:
# merge(): 특정 컬럼(키) 값이 일치하는 것끼리 정확하게 매칭

print(orders.shape)
print(customers.shape)

# default: how = 'inner'




| 구분 | `concat()` | `merge()` |
|---|---|---|
| 결합 기준 | 기준 없이 단순히 이어붙임(순서/축 기준) | 특정 컬럼(키) 값이 일치하는 것끼리 정교하게 매칭 |
| 주 용도 | 같은 구조의 데이터를 누적(월별 매출 등) | 서로 다른 정보를 가진 표를 연결(주문+고객 정보 등) |
| 대응하는 SQL 개념 | UNION과 유사 | JOIN과 유사 |
| 행 개수 변화 | 단순 합산 | 매칭 방식(how)에 따라 달라짐 |

> 💡 **핵심 요약**: "같은 형태의 데이터를 계속 쌓아야 한다"면 `concat()`, "서로 다른 정보를 공통된 키로 연결해야 한다"면 `merge()`를 사용하세요.

---

## ⚠️ 자주 하는 실수

> ⚠️ `concat()`으로 서로 다른 컬럼 구조의 표를 이어붙이면, 없는 값들이 자동으로 `NaN`으로 채워지는 것을 인지하지 못하는 경우가 있습니다.

> ⚠️ `merge()`의 기본값이 `inner`라는 것을 모르고, 예상보다 행이 적게 나와서 데이터가 사라졌다고 착각하는 경우가 흔합니다.

> ⚠️ 병합 기준 컬럼의 값 표기가 미묘하게 다르면(예: "개발팀" vs "개발팀 "처럼 공백 차이) 매칭이 안 되는 경우가 있습니다.

---

## 🚀 실무에서는?

> 🚀 실무 데이터 분석은 "여러 테이블을 합치는 것"에서 시작하는 경우가 대부분입니다. 회원 테이블과 주문 테이블을 `merge()`로 합쳐 "회원별 총 구매액"을 구하거나, 여러 달의 매출 파일을 `concat()`으로 합쳐 연간 데이터를 만드는 작업이 매우 흔합니다.

---

## 🧩 Check Point

**Q1.** 양쪽 표에 공통으로 있는 키만 결과에 남기는 병합 방식은? ① left ② right ③ inner ④ outer
<details><summary>정답</summary>③ inner</details>

**Q2.** 같은 구조의 데이터를 위아래로 단순히 쌓을 때 사용하는 함수는? ① merge() ② concat() ③ join() ④ pivot_table()
<details><summary>정답</summary>② concat()</details>

---

## 💻 실습

> 🔧 아래 코드 블록은 ipynb 셀로 그대로 변환됩니다. 난이도가 단계적으로 올라갑니다.

In [ ]:
# TODO 1 (concat): orders와 orders_extra.csv를 세로로 합치고 인덱스를 재정렬하세요

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
combined = pd.concat([orders, extra], ignore_index=True)
print(combined.tail())    # 7월 주문(2001~)이 아래에 붙음
```

</details>

In [ ]:
# 코드 입력





In [ ]:
# TODO 2 (merge 기본): orders와 customers를 "고객ID" 기준으로 병합하세요 (기본 inner)

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
print(pd.merge(orders, customers, on="고객ID").head())
```

</details>

In [ ]:
# 코드 입력





In [ ]:
# TODO 3 (merge how 비교): 같은 병합을 how="left"와 how="outer"로 각각 수행하고,
#         행 개수가 왜 다른지 확인하세요 (힌트: C999와 6월 주문이 없는 고객들)

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
left = pd.merge(orders, customers, on="고객ID", how="left")
outer = pd.merge(orders, customers, on="고객ID", how="outer")
print("left:", left.shape)      # 주문 300건 전부 유지
print("outer:", outer.shape)    # 6월 주문이 없는 고객 3명까지 포함되어 3행 더 많음
```

</details>

In [ ]:
# 코드 입력





In [ ]:
# TODO 4 (pivot_table): 카테고리별/결제수단별 주문 수량 합계 피벗테이블을 만드세요

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
print(orders.pivot_table(index="카테고리", columns="결제수단코드",
                         values="수량", aggfunc="sum"))
```

</details>

In [ ]:
# 코드 입력





In [ ]:
# TODO 5 (json_normalize): 중첩된 주문 JSON 레코드 리스트를 평평한 DataFrame으로 변환하세요

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
records = [{"주문번호": 3001, "고객": {"고객ID": "C001", "등급": "VIP"}}]
print(pd.json_normalize(records))
```

</details>

In [ ]:
# 코드 입력





---

## 📝 실습 과제

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 과제 1: orders(6월)와 orders_extra(7월)를 concat으로 합친 뒤,
#         주문일을 datetime으로 변환해 월별 주문 건수를 구하세요

# 과제 2: orders와 customers를 merge()로 합치되, 고객 명단에 없는 주문(C999)도 유지되도록 하고,
#         고객명이 비어있는(매칭 실패) 주문을 찾아 출력하세요

# 과제 3: {"주문번호": 4001, "고객": {"이름": "김하늘", "등급": "VIP"}} 형태의 중첩 딕셔너리 3개를
#         json_normalize()로 DataFrame으로 만드세요
```

</details>

In [ ]:
# 과제 1: orders(6월)와 orders_extra(7월)를 concat으로 합친 뒤,
#         주문일을 datetime으로 변환해 월별 주문 건수를 구하세요

In [ ]:
# 과제 2: orders와 customers를 merge()로 합치되, 고객 명단에 없는 주문(C999)도 유지되도록 하고,
#         고객명이 비어있는(매칭 실패) 주문을 찾아 출력하세요

In [ ]:
# 과제 3: {"주문번호": 4001, "고객": {"이름": "김하늘", "등급": "VIP"}} 형태의 중첩 딕셔너리 3개를
#         json_normalize()로 DataFrame으로 만드세요

**예시 정답**

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 과제 1 예시 정답
all_orders = pd.concat([orders, extra], ignore_index=True)
all_orders["주문일"] = pd.to_datetime(all_orders["주문일"])
print(all_orders["주문일"].dt.month.value_counts().sort_index())   # 6월 300건, 7월 40건
```

</details>

In [ ]:
# 과제 1 예시 정답




<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 과제 2 예시 정답
merged = pd.merge(orders, customers, on="고객ID", how="left")
print(merged[merged["고객명"].isnull()][["주문번호", "고객ID", "상품명"]])
# C999 고객의 주문 — 명단 누락인지 비회원 주문인지 확인이 필요한 케이스
```

</details>

In [ ]:
# 과제 2 예시 정답




<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 과제 3 예시 정답
records = [
    {"주문번호": 4001, "고객": {"이름": "김하늘", "등급": "VIP"}},
    {"주문번호": 4002, "고객": {"이름": "이준서", "등급": "일반"}},
    {"주문번호": 4003, "고객": {"이름": "최은우", "등급": "신규"}},
]
print(pd.json_normalize(records))
```

</details>

In [ ]:
# 과제 3 예시 정답





---

## 📌 핵심 정리

- `concat()`: 같은 구조의 데이터를 단순히 이어붙임 (세로 axis=0, 가로 axis=1)
- `merge(on=, how=)`: 특정 컬럼(키) 기준으로 정교하게 결합, how는 inner/left/right/outer
- `join()`: 인덱스를 기준으로 병합
- `pivot_table()`: 행/열 기준을 지정해 집계, `margins=True`로 합계 추가
- `pd.json_normalize()`: 중첩된 JSON을 평평한 표로 변환, `to_json()`/`read_json()`으로 파일 입출력

---

---

# 📘 CH09-05. apply() 활용

## 🤔 먼저 생각해보기

> **주문마다 "주문금액(단가 × 수량)"을 계산하고, 5만원 이상이면 "고액주문"으로 분류하는 컬럼을 만들고 싶다면, 한 줄 한 줄 반복문을 돌려야 할까요?**

Pandas의 `apply()`를 사용하면 반복문 없이 **함수를 데이터 전체에 한 번에 적용**할 수 있습니다.

---

## 📖 핵심 개념

### 0) 실습 데이터 준비 (지난 소단원 전처리 복습 포함)

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# 지난 소단원 복습: 계산 가능하도록 전처리 먼저!
df["단가"] = df["단가"].str.replace(",", "").astype(int)
df = df.drop_duplicates()
df = df.dropna(subset=["수량"])
print(df[["상품명", "단가", "수량"]].head())
```

</details>

In [ ]:
# 코드 입력

import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# 지난 소단원 복습: 계산 가능하도록 전처리 먼저!

# 금액에 콤마제거후 int 변환
df["단가"] = df["단가"].str.replace(",", "").astype(int)

# 중복 제거
df = df.drop_duplicates()
# 결측치 제거 ('수량' 컬럼에 결측치만)
df = df.dropna(subset=["수량"])


print(df[["상품명", "단가", "수량"]].head())

### 1) `apply()`란? — Series/DataFrame 전체에 함수 적용

### 2) Series에 `apply()` 적용하기

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
def classify_price(price):
    if price >= 40000:
        return "고가"
    elif price >= 15000:
        return "중가"
    else:
        return "저가"

df["가격대"] = df["단가"].apply(classify_price)    # 각 단가 값에 함수가 하나씩 적용됨
print(df[["상품명", "단가", "가격대"]].head())
```

</details>

In [ ]:
# 코드 입력




# 각 단가 값에 함수가 하나씩 적용됨
# apply는 마치 for문 처럼 순회하며 반환





<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# lambda와 함께 쓰면 더 간결
df["가격대2"] = df["단가"].apply(lambda x: "고가" if x >= 40000 else "일반")
print(df[["상품명", "단가", "가격대2"]].head())
```

</details>

In [ ]:
# lambda와 함께 쓰면 더 간결
# 고가와 일반처럼 간단한 조건일 경우 사용





<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 간단한 계산이라면 함수 없이도 사용 가능
df["할인단가"] = df["단가"].apply(lambda x: x * 0.9)    # 10% 할인가
print(df[["상품명", "단가", "할인단가"]].head())
```

</details>

In [ ]:
# 간단한 계산이라면 함수 없이도 사용 가능
# 10% 할인가




### 3) DataFrame에 `apply()` 적용하기 — `axis` 옵션

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# axis=0 (기본값): 각 "열(컬럼)"에 함수를 적용
print(df[["단가", "수량"]].apply(lambda col: col.max()))    # 컬럼별 최댓값
```

</details>

In [ ]:
# axis=0 (기본값) 생략가능
# 각 "열(컬럼)"에 함수를 적용

# 컬럼별 최댓값





<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# axis=1: 각 "행"에 함수를 적용 (행 전체를 하나의 Series로 받음)
def order_amount(row):
    return row["단가"] * row["수량"]        # 한 행 안의 두 컬럼 값을 조합!

df["주문금액"] = df.apply(order_amount, axis=1)
print(df[["상품명", "단가", "수량", "주문금액"]].head())
```

</details>

In [ ]:
# axis=1: 각 "행"에 함수를 적용 (행 전체를 하나의 Series로 받음)





<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# axis=1 + lambda로 여러 컬럼을 조합한 계산
df["주문요약"] = df.apply(lambda row: f'{row["상품명"]} {int(row["수량"])}개 = {int(row["주문금액"]):,}원', axis=1)
print(df["주문요약"].head())
```

</details>

In [127]:
# axis=1 + lambda로 여러 컬럼을 조합한 계산
# 출력 예시: 무릎담요 1개 = 22,000원




0    무릎담요 1개 = 22,000원
1    양말세트 3개 = 26,700원
2      우산 3개 = 45,000원
3      향초 3개 = 28,500원
4     디퓨저 3개 = 49,500원
Name: 주문요약, dtype: object


### 4) 여러 값을 반환해서 새 컬럼 여러 개 만들기

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
def amount_analysis(amount):
    vat = amount * 0.1               # 부가세 10%
    total = amount + vat             # 부가세 포함 금액
    return pd.Series([vat, total])

df[["부가세", "결제금액"]] = df["주문금액"].apply(amount_analysis)
print(df[["상품명", "주문금액", "부가세", "결제금액"]].head())
```

</details>

In [ ]:
# 코드 입력





### 5) 서로 다른 방식 비교 — `apply(axis=0)` vs `apply(axis=1)`

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# axis=0: 컬럼 단위로 함수가 호출됨 (컬럼 전체가 함수의 인자로 전달)
result_col = df[["단가", "수량"]].apply(lambda col: col.mean())
print(result_col)    # 단가 평균, 수량 평균
```

</details>

In [ ]:
# axis=0: 컬럼 단위로 함수가 호출됨 (컬럼 전체가 함수의 인자로 전달)





<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# axis=1: 행 단위로 함수가 호출됨 (한 행 전체가 함수의 인자로 전달)
result_row = df[["단가", "수량"]].apply(lambda row: row["단가"] * row["수량"], axis=1)
print(result_row.head())    # 행마다 단가×수량
```

</details>

In [ ]:
# axis=1: 행 단위로 함수가 호출됨 (한 행 전체가 함수의 인자로 전달)





| 구분 | `axis=0` (기본값) | `axis=1` |
|---|---|---|
| 함수에 전달되는 단위 | 컬럼(세로줄) 하나 전체 | 행(가로줄) 하나 전체 |
| 주로 하는 일 | 컬럼별 통계 계산 (컬럼 간 독립적 계산) | 여러 컬럼 값을 조합한 계산 (같은 행 안의 값들 활용) |
| 예시 | 컬럼별 평균, 최댓값-최솟값 | "단가와 수량을 곱한 주문금액 계산" |

> 💡 **핵심 요약**: "각 컬럼끼리 독립적으로" 계산하려면 `axis=0`(기본값), "한 행 안에 있는 여러 컬럼 값을 함께" 활용해야 한다면 `axis=1`을 사용하세요.

### 6) 서로 다른 방식 비교 — `apply()` vs 벡터 연산

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
import time
import numpy as np

big_df = pd.DataFrame({"값": np.arange(1, 500001)})

# 방법 A: apply() — 내부적으로 사실상 반복문과 유사하게 동작
start = time.time()
result_apply = big_df["값"].apply(lambda x: x * 2)
print("apply 소요시간:", time.time() - start)

# 방법 B: 벡터 연산 — NumPy/Pandas의 최적화된 연산
start = time.time()
result_vector = big_df["값"] * 2
print("벡터 연산 소요시간:", time.time() - start)
```

</details>

In [ ]:
# 코드 입력

import time
import numpy as np

big_df = pd.DataFrame({"값": np.arange(1, 500001)})

# 방법 A: apply() — 내부적으로 사실상 반복문과 유사하게 동작




# 방법 B: 벡터 연산 — NumPy/Pandas의 최적화된 연산






In [ ]:
# 참고




### 7) 조건 분기를 빠르게 — `np.where()`

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
import numpy as np

# apply() 방식: 각 값마다 파이썬 함수 호출 (느림)
df["주문유형_apply"] = df["주문금액"].apply(lambda x: "고액주문" if x >= 50000 else "일반주문")

# np.where() 방식: 조건 전체를 벡터 연산으로 한 번에 (빠름)
df["주문유형_where"] = np.where(df["주문금액"] >= 50000, "고액주문", "일반주문")

print(df[["상품명", "주문금액", "주문유형_apply", "주문유형_where"]].head())
```

</details>

In [ ]:
# 코드 입력

import numpy as np




> 💡 `np.where(조건, 참일 때 값, 거짓일 때 값)`은 "조건에 따라 둘 중 하나"를 고르는 작업을 벡터 연산으로 처리해, 같은 일을 하는 `apply()`보다 훨씬 빠릅니다. 단순한 2분기 조건이라면 `np.where()`를 먼저 고려하고, 3개 이상의 복잡한 분기나 여러 컬럼을 조합하는 로직은 `apply()`가 더 읽기 편합니다.

| 구분 | `apply()` | 벡터 연산 (`*`, `+` 등) |
|---|---|---|
| 처리 방식 | 내부적으로 각 값에 대해 파이썬 함수를 반복 호출 | 최적화된 내부 연산으로 한 번에 처리 |
| 속도 | 상대적으로 느림 | 훨씬 빠름 |
| 표현력 | 복잡한 조건/로직도 자유롭게 표현 가능 | 단순한 사칙연산·비교 등에 적합 |
| 언제 사용? | 조건 분기 등 벡터 연산으로 표현하기 어려운 로직 | 단순 계산은 항상 벡터 연산 우선 고려 |

> 💡 **핵심 요약**: `apply()`는 편리하지만 "느린 반복문"에 가깝습니다. 단순한 계산이라면 벡터 연산으로 먼저 시도해보고, 조건이 복잡해서 벡터 연산으로 표현하기 어려울 때만 `apply()`를 사용하는 것이 성능 면에서 유리합니다.

---

## ⚠️ 자주 하는 실수

> ⚠️ 단순한 사칙연산까지 습관적으로 `apply()`로 처리해서 불필요하게 느려지는 경우가 흔합니다 (`df["단가"] * 1.1`이면 충분한데 `apply(lambda x: x*1.1)`을 사용).

> ⚠️ 단순 2분기 조건("~이면 A, 아니면 B")까지 `apply()`로 처리하는 경우가 많습니다 — `np.where()`가 더 빠르고 간결합니다.

> ⚠️ `axis` 옵션을 빼먹고 기본값(0)이 컬럼 단위로 동작한다는 것을 잊어, 행 단위 계산이 필요한데 엉뚱한 결과가 나오는 경우가 있습니다.

> ⚠️ `apply()`에 전달하는 함수가 항상 값을 `return`하는지 확인하지 않아 결과가 전부 `None`이 되는 경우가 있습니다.

---

## 🚀 실무에서는?

> 🚀 데이터의 여러 컬럼을 조합해 새로운 파생 변수를 만들거나(예: 나이와 소득을 함께 고려한 고객 등급), 복잡한 조건에 따라 값을 분류할 때 `apply(axis=1)`이 자주 사용됩니다. 다만 대용량 데이터에서는 성능을 위해 가능한 벡터 연산으로 대체하려는 시도가 함께 이루어집니다.

---

## 🧩 Check Point

**Q1.** DataFrame에서 각 "행"에 함수를 적용하고 싶을 때 지정하는 옵션은? ① axis=0 ② axis=1 ③ axis=2 ④ 옵션 불필요
<details><summary>정답</summary>② axis=1</details>

**Q2.** 대량의 단순 사칙연산에 더 빠른 방법은? ① apply() ② 벡터 연산
<details><summary>정답</summary>② 벡터 연산</details>

---

## 💻 실습

> 🔧 아래 코드 블록은 ipynb 셀로 그대로 변환됩니다. 난이도가 단계적으로 올라갑니다.

In [ ]:
# TODO 1 (Series apply): 단가에 apply()로 5% 인상가를 계산한 "인상단가" 컬럼을 만드세요

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
df["인상단가"] = df["단가"].apply(lambda x: x * 1.05)
print(df[["상품명", "단가", "인상단가"]].head())
```

</details>

In [ ]:
# 코드 입력




In [ ]:
# TODO 2 (조건 함수): 수량이 3 이상이면 "다량", 미만이면 "소량"으로 분류하는 컬럼을 만드세요

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
df["수량구분"] = df["수량"].apply(lambda x: "다량" if x >= 3 else "소량")
print(df[["상품명", "수량", "수량구분"]].head())
```

</details>

In [ ]:
# 코드 입력




In [ ]:
# TODO 3 (axis=1): 수량 3 이상이면서 주문금액 50000 이상이면 "우수주문",
#         아니면 "일반주문"으로 분류하는 컬럼을 apply(axis=1)로 만드세요

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
df["주문구분"] = df.apply(lambda row: "우수주문" if row["수량"] >= 3 and row["주문금액"] >= 50000 else "일반주문", axis=1)
print(df[["상품명", "수량", "주문금액", "주문구분"]].head())
```

</details>

In [ ]:
# 코드 입력




In [ ]:
# TODO 4 (axis=0): 단가와 주문금액 컬럼 각각의 표준편차를 apply()로 구하세요

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
print(df[["단가", "주문금액"]].apply(lambda col: col.std()))
```

</details>

In [ ]:
# 코드 입력




In [ ]:
# TODO 5 (여러 값 반환): 주문금액을 받아 (부가세, 결제금액) 두 값을 반환하는 함수를 만들어
#        두 개의 새 컬럼을 한 번에 생성하세요

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
def vat_calc(amount):
    vat = amount * 0.1
    return pd.Series([vat, amount + vat])

df[["부가세2", "결제금액2"]] = df["주문금액"].apply(vat_calc)
print(df[["주문금액", "부가세2", "결제금액2"]].head())
```

</details>

In [ ]:
# 코드 입력




---

## 📝 실습 과제

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 과제 1: 상품명의 글자 수를 세는 "상품명길이" 컬럼을 apply()로 추가하세요 (힌트: len 함수 활용)

# 과제 2: 수량, 주문금액, 배송평점 세 조건을 모두 고려해
#         "우수/보통/저조" 3단계로 평가하는 함수를 만들어 apply(axis=1)로 적용하세요

# 과제 3: 단가에 단순히 10%를 더하는 작업을 apply()와 벡터 연산으로 각각 수행하고
#         100만 개 데이터 기준으로 속도를 비교하세요 (time 모듈 활용)
```

</details>

In [ ]:
# 과제 1: 상품명의 글자 수를 세는 "상품명길이" 컬럼을 apply()로 추가하세요 (힌트: len 함수 활용)

In [ ]:
# 과제 2: 수량, 주문금액, 배송평점 세 조건을 모두 고려해
#         "우수/보통/저조" 3단계로 평가하는 함수를 만들어 apply(axis=1)로 적용하세요
#         (예: 금액 5만 이상 & 평점 4 이상 → 우수, 평점 2 이하 → 저조, 나머지 → 보통)

In [ ]:
# 과제 3: 단가에 단순히 10%를 더하는 작업을 apply()와 벡터 연산으로 각각 수행하고
#         100만 개 데이터 기준으로 속도를 비교하세요 (time 모듈 활용)

**예시 정답**

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 과제 1 예시 정답
df["상품명길이"] = df["상품명"].apply(len)
print(df[["상품명", "상품명길이"]].head())
```

</details>

In [ ]:
# 과제 1 예시 정답




<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 과제 2 예시 정답
def evaluate_order(row):
    if row["주문금액"] >= 50000 and row["배송평점"] >= 4:
        return "우수"
    elif row["배송평점"] <= 2:
        return "저조"
    else:
        return "보통"

df["주문평가"] = df.apply(evaluate_order, axis=1)
print(df[["상품명", "주문금액", "배송평점", "주문평가"]].head(10))
```

</details>

In [ ]:
# 과제 2 예시 정답





<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 과제 3 예시 정답
import time
import numpy as np
big = pd.DataFrame({"단가": np.arange(1, 1_000_001)})

start = time.time()
r1 = big["단가"].apply(lambda x: x * 1.1)
print("apply 방식:", round(time.time() - start, 3), "초")

start = time.time()
r2 = big["단가"] * 1.1
print("벡터 연산:", round(time.time() - start, 3), "초")
```

</details>

In [ ]:
# 과제 3 예시 정답

import time
import numpy as np





---

## 📌 핵심 정리

- `Series.apply(함수)`: 각 값에 함수 적용
- `DataFrame.apply(함수, axis=0)`: 각 컬럼에 함수 적용(기본값) / `axis=1`: 각 행에 함수 적용
- 함수가 여러 값을 `pd.Series([...])`로 반환하면 여러 개의 새 컬럼으로 한 번에 추가 가능
- 단순 계산은 벡터 연산이 훨씬 빠름 — `apply()`는 벡터 연산으로 표현하기 어려운 복잡한 로직에 사용
- `np.where(조건, A, B)`: 단순 2분기 조건은 apply()보다 빠른 벡터 방식 우선

---

---
## 🎉 정규 수업 완료!

결측치, 이상치, 자료형 변환, 데이터 병합, apply() 5개 소단원을 실제 CSV 파일(`orders.csv`, `customers.csv`, `orders_extra.csv`) 기반으로 모두 다뤘습니다. 한 파일 안의 결측치·이상치·공백·중복·자료형 문제를 순서대로 해결하며 전처리 전체 흐름을 경험했습니다. 자율학습 심화 내용은 CH09 관련 md 문서를 참고해주세요.
